# TonyPi PnP 定位测试

本示例用于验证实验三中的 AprilTag 定位方法。程序读取 TonyPi 项目已有的场地 Tag 世界坐标和相机标定参数，识别一帧图像中的 AprilTag，再通过 solvePnP 计算相机的位置与朝向。

运行前请让机器人保持静止、头部朝向正前方，并确保画面中至少有一个完整清晰的 AprilTag。

## 1. 导入库并设置路径

本 Notebook 默认 TonyPi 项目位于 /home/pi/robot_tonypi，机器人硬件库位于 /home/pi/TonyPi。

In [ ]:
import sys
import time
from pathlib import Path

import apriltag
import cv2
import numpy as np
from IPython.display import Image, display

TONYPI_ROOT = Path('/home/pi/TonyPi')
ROBOT_TONYPI_ROOT = Path('/home/pi/robot_tonypi')
CALIBRATION_FILE = ROBOT_TONYPI_ROOT / 'camera_distortion_calibration' / 'calibration_param.npz'

for path in (TONYPI_ROOT, TONYPI_ROOT / 'HiwonderSDK', ROBOT_TONYPI_ROOT):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

import hiwonder.Camera as Camera
from load_pos import load_tag_pos

print('库和路径加载完成')

## 2. 加载场地地图和相机标定参数

load_tag_pos() 提供各 AprilTag 四个角点的世界坐标。相机内参与畸变系数直接读取 robot_tonypi/camera_distortion_calibration/calibration_param.npz。

In [ ]:
if not CALIBRATION_FILE.exists():
    raise FileNotFoundError(f'未找到相机标定文件：{CALIBRATION_FILE}')

tag_world_points = load_tag_pos()
calibration = np.load(CALIBRATION_FILE)
camera_matrix = np.asarray(calibration['mtx_array'], dtype=np.float64)
dist_coeffs = np.asarray(calibration['dist_array'], dtype=np.float64).reshape(-1)

print(f'已加载 {len(tag_world_points)} 个 Tag 的世界坐标')
print('相机内参：')
print(camera_matrix)
print('畸变系数：', dist_coeffs)

## 3. 创建 AprilTag 检测器并打开相机

实验场地使用 tag36h11 标签。相机只需打开一次，测试完成后请运行最后的关闭单元。

In [ ]:
detector = apriltag.Detector(apriltag.DetectorOptions(families='tag36h11'))
camera = Camera.Camera()
camera.camera_open()
time.sleep(1)
print('检测器和相机已准备完成')

## 4. 定义 PnP 定位函数

函数将已知 Tag 的世界角点与像素角点组成对应关系。solvePnP 得到旋转和平移后，通过 -Rᵀt 计算相机的世界坐标，并根据相机光轴判断朝向。

In [ ]:
def show_image(frame):
    # 在 Notebook 中显示 OpenCV 图像。
    ok, encoded = cv2.imencode('.jpg', frame)
    if ok:
        display(Image(data=encoded.tobytes()))


def direction_name(yaw_deg):
    # 将角度转换为实验要求的四个基本朝向。
    if -45.0 <= yaw_deg < 45.0:
        return 'x 轴正方向'
    if 45.0 <= yaw_deg < 135.0:
        return 'y 轴正方向'
    if yaw_deg >= 135.0 or yaw_deg < -135.0:
        return 'x 轴负方向'
    return 'y 轴负方向'

In [ ]:
def collect_correspondences(frame):
    # 收集地图世界角点和图像像素角点。
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    detections = detector.detect(gray)
    annotated = frame.copy()
    object_points = []
    image_points = []
    used_ids = []

    for tag in detections:
        tag_id = str(tag.tag_id)
        corners = np.asarray(tag.corners, dtype=np.float64)
        color = (0, 255, 0) if tag_id in tag_world_points else (0, 0, 255)
        cv2.polylines(annotated, [np.round(corners).astype(np.int32)], True, color, 2)
        center = tuple(np.round(tag.center).astype(int))
        cv2.putText(annotated, 'ID: ' + tag_id, (center[0] - 25, center[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        if tag_id in tag_world_points:
            object_points.extend(np.asarray(tag_world_points[tag_id], dtype=np.float64))
            image_points.extend(corners)
            used_ids.append(int(tag_id))

    return (np.asarray(object_points, dtype=np.float64),
            np.asarray(image_points, dtype=np.float64), annotated, used_ids)

In [ ]:
def locate_with_pnp(frame):
    # 通过 PnP 计算相机的世界位置和朝向。
    object_points, image_points, annotated, used_ids = collect_correspondences(frame)
    if len(object_points) < 4:
        return None, annotated, used_ids

    success, rvec, tvec = cv2.solvePnP(
        object_points, image_points, camera_matrix, dist_coeffs
    )
    if not success:
        return None, annotated, used_ids

    rotation_matrix, _ = cv2.Rodrigues(rvec)
    camera_position = -rotation_matrix.T @ tvec
    camera_forward = rotation_matrix.T @ np.array([[0.0], [0.0], [1.0]])
    yaw_deg = float(np.degrees(np.arctan2(camera_forward[1, 0], camera_forward[0, 0])))

    result = {
        'x_cm': float(camera_position[0, 0]),
        'y_cm': float(camera_position[1, 0]),
        'z_cm': float(camera_position[2, 0]),
        'yaw_deg': yaw_deg,
        'direction': direction_name(yaw_deg),
    }
    return result, annotated, used_ids

## 5. 拍摄一帧并计算位置

运行本单元后会输出参与计算的 Tag ID、相机的世界坐标和朝向。地图坐标单位为厘米，因此输出位置同样以厘米为单位。若未定位成功，请调整机器人位置，使 Tag 完整出现在画面中后再次运行。

In [ ]:
ret, frame = False, None
for _ in range(5):
    ret, frame = camera.read()
    time.sleep(0.05)

if not ret or frame is None:
    print('未获取到相机画面，请检查相机连接。')
else:
    pose, annotated, used_ids = locate_with_pnp(frame)
    print('参与 PnP 计算的 Tag ID：', used_ids)
    if pose is None:
        print('定位失败：没有识别到地图中的完整 AprilTag，或 solvePnP 计算失败。')
    else:
        print('相机位置：x={:.1f} cm, y={:.1f} cm, z={:.1f} cm'.format(
            pose['x_cm'], pose['y_cm'], pose['z_cm']))
        print('相机朝向：{:.1f}°，接近{}'.format(pose['yaw_deg'], pose['direction']))
    show_image(annotated)

## 6. 关闭相机

完成测试后运行本单元，释放相机设备。

In [ ]:
camera.camera_close()
print('相机已关闭')